In [1]:
import pandas as pd
df = pd.read_csv("telecom_churn_cleaned.csv")
df.head()

,Customer ID,SeniorCitizen_Status,Contract,Churn,Payment Method,MonthlyCharges_Clean,Tenure,TotalCharges_Clean
0,0002-ORFBO,No,One year,No,Mailed check,0,9,593
1,0003-MKNFE,No,Month-to-month,No,Mailed check,0,9,542
2,0004-TLHLJ,No,Month-to-month,Yes,Electronic check,0,4,280
3,0011-IGKFF,Yes,Month-to-month,Yes,Electronic check,98,13,"1,237"
4,0013-EXCHZ,Yes,Month-to-month,Yes,Mailed check,0,3,267


In [2]:
df.isnull().sum()

Customer ID             0
SeniorCitizen_Status    0
Contract                0
Churn                   0
Payment Method          0
MonthlyCharges_Clean    0
Tenure                  0
TotalCharges_Clean      0
dtype: int64

In [3]:
# train test split
from sklearn.model_selection import train_test_split
x = df[['Tenure', 'MonthlyCharges_Clean', 'TotalCharges_Clean']]
y = df['Churn']
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
print("Training data size:",len(x_train))
print("Testing data size:",len(x_test))


Training data size: 5634
Testing data size: 1409


In [4]:
#convert TotalCharges_Clean to numeric
x_train['TotalCharges_Clean'] = pd.to_numeric(x_train['TotalCharges_Clean'], errors='coerce')
x_test['TotalCharges_Clean'] = pd.to_numeric(x_test['TotalCharges_Clean'], errors='coerce')


In [5]:
print(x_train.isnull().sum())

Tenure                     0
MonthlyCharges_Clean       0
TotalCharges_Clean      3299
dtype: int64


In [6]:
# fill nan with median in total charges
x_train['TotalCharges_Clean'].fillna(x_train['TotalCharges_Clean'].median(), inplace=True)
x_test['TotalCharges_Clean'].fillna(x_test['TotalCharges_Clean'].median(), inplace=True)
print(x_train.isnull().sum())

Tenure                  0
MonthlyCharges_Clean    0
TotalCharges_Clean      0
dtype: int64


C:\Windows\Temp\ipykernel_11836\2447616889.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  x_train['TotalCharges_Clean'].fillna(x_train['TotalCharges_Clean'].median(), inplace=True)
C:\Windows\Temp\ipykernel_11836\2447616889.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

F

In [7]:
# logistic regression model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(x_train,y_train)
print("model trained successfully")

model trained successfully


In [8]:
# calculate m and c for the logistic regression equation
m = model.coef_[0]
c = model.intercept_[0]
print("m:",m)
print("c:",c)

m: [-0.03860822  0.00519283 -0.00056206]
c: 0.1763094449201356


In [9]:
# calculate z value
import math
import numpy as np
x = [5, 70, 3500]
z = m[0]*x[0] + m[1]*x[1] + m[2]*x[2] + c
print(z)
probability = 1/(1+np.exp(-z))
print(probability)

if (probability>0.6):
    print("YES CHURN")
else:
    print("NO CHURN")


y_pred = model.predict(x_test)
print(y_pred)

comparision = pd.DataFrame({"Actual": y_test.values.flatten(), "Predicted": y_pred.flatten()})
print(comparision)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,y_pred)
print(cm)

-1.6204318799468396
0.16514531727967507
NO CHURN
['No' 'No' 'No' ... 'Yes' 'No' 'No']
     Actual Predicted
0        No        No
1        No        No
2        No        No
3        No        No
4        No        No
...     ...       ...
1404    Yes        No
1405     No        No
1406    Yes       Yes
1407     No        No
1408     No        No

[1409 rows x 2 columns]
[[955  81]
 [281  92]]


In [10]:
# classification report
from sklearn.metrics import classification_report
cr = classification_report(y_test,y_pred)
print(cr)

              precision    recall  f1-score   support

          No       0.77      0.92      0.84      1036
         Yes       0.53      0.25      0.34       373

    accuracy                           0.74      1409
   macro avg       0.65      0.58      0.59      1409
weighted avg       0.71      0.74      0.71      1409

